# Q4. Net PO value by material group and dominant suppliers

Scope (from `eda/scope_eda`): purchasing org **US00**, purchase orders only (category F, deleted items excluded), Document Date **Oct 2025 – Sep 2026**, USD. Measure = **Net PO value** (ordered value, not invoiced or paid spend).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data and charts

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sap_data import load_scope, product_family, FAMILY_CATEGORY
from charts import SERIES, MUTED, INK_2, usd, pct, style, save

style()
pd.set_option("display.max_columns", 30)
PERIOD = "Oct 2025 – Sep 2026"

w = load_scope()
w["supplier"] = w["Name 1"] + " (" + w["Supplier"].fillna("n/a") + ")"
TOTAL = w["Net Order Value"].sum()
print(f"{len(w)} PO items | {w['Purchasing Document'].nunique()} POs | {TOTAL:,.0f} USD")

**Question.** What is the breakdown of Net PO value by material group, and which suppliers dominate the
critical, high-performance categories?

SAP material groups in scope are only `UTIL`, `RAW` and `BIKES`, so they don't map onto Frames/Wheels,
Off-road components or Accessories. We therefore also use a **derived product family**, taken from
the item text and the material prefix (`sap_data.product_family`), and group the families into
categories:

| Category | Product families |
|---|---|
| Accessories | chain lock, water bottle, bottle cage, t-shirt |
| Components | socket head bolt, off-road handle bar |
| Finished bikes | touring bikes |
| Non-bike (exercise item) | spelt flour, a student test material |

In [ ]:
w["family"] = product_family(w)
w["category"] = w["family"].map(FAMILY_CATEGORY).fillna("Other")

def shares(col):
    return (w.groupby(col, dropna=False)["Net Order Value"].agg(net="sum", items="size")
              .assign(share=lambda d: d.net / d.net.sum()).sort_values("net", ascending=False))

mg, cat = shares("Material Group"), shares("category")
display(mg.style.format({"net": "{:,.0f}", "share": "{:.1%}"}))
display(cat.style.format({"net": "{:,.0f}", "share": "{:.1%}"}))
shares("family").style.format({"net": "{:,.0f}", "share": "{:.1%}"})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1, 1.3]})
for ax, d, title in [(axes[0], mg, "SAP material group"), (axes[1], cat, "Derived category")]:
    d = d[d["net"] > 0].iloc[::-1]
    names = d.index.fillna("(blank)").astype(str)
    ax.barh(names, d["share"], color=SERIES[0], height=0.6)
    for y, (v, n) in enumerate(zip(d["share"], d["net"])):
        ax.text(v, y, f"  {v:.1%} · ${n / 1e3:,.0f}k", va="center", color=INK_2, fontsize=9)
    ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1.3); ax.set_xticks([0, .25, .5, .75, 1]); ax.set_title(title)
    ax.grid(axis="y", visible=False)
fig.suptitle(f"Share of Net PO value by material, US00, {PERIOD}", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "q4_material_breakdown")

### Dominant suppliers per category
Concentration inside each category: the top supplier's share, the top-5 share and the number of
suppliers.

In [ ]:
cats = ["Accessories", "Components", "Finished bikes"]
rows = []
for c in cats:
    s = w[w["category"] == c].groupby("supplier")["Net Order Value"].sum().sort_values(ascending=False)
    sh = s / s.sum()
    rows.append({"category": c, "net": s.sum(), "suppliers": len(s), "top1_supplier": s.index[0],
                 "top1_share": sh.iat[0], "top5_share": sh.head(5).sum(), "HHI": (sh ** 2).sum()})
pd.DataFrame(rows).set_index("category").style.format(
    {"net": "{:,.0f}", "top1_share": "{:.0%}", "top5_share": "{:.0%}", "HHI": "{:.3f}"})

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, c in zip(axes, cats):
    s = w[w["category"] == c].groupby("supplier")["Net Order Value"].sum().sort_values(ascending=False)
    top = (s.head(5) / s.sum()).iloc[::-1]
    ax.barh(top.index.str.slice(0, 32), top, color=SERIES[0], height=0.6)
    for y, v in enumerate(top):
        ax.text(v, y, f" {v:.0%}", va="center", color=INK_2, fontsize=9)
    ax.xaxis.set_major_formatter(pct); ax.set_xlim(0, 1.15); ax.set_xticks([0, .25, .5, .75, 1])
    ax.set_title(f"{c}  (${s.sum() / 1e3:,.0f}k, {len(s)} suppliers)", fontsize=10.5)
    ax.grid(axis="y", visible=False)
fig.suptitle("Top 5 suppliers' share of each category's Net PO value", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "q4_top_suppliers_by_category")

## Findings for Antonio

**Finding.** SAP material groups are too coarse for the question: `UTIL` holds **94.7%**, `BIKES` 2.9% and `RAW` 2.4%. Frames, wheels and off-road components don't exist as groups in this window. Using derived categories, **Accessories are 74.8%** (chain locks alone are 64.8% of all value), finished bikes 2.9% and components 0.4%. 21.9% is a non-bike exercise item (spelt flour). Concentration differs a lot by category:
- **Finished bikes:** 3 suppliers, and the top one (szállító-032) has **85%** (HHI 0.74). This is high dependence.
- **Components:** 104 suppliers, top supplier 19%, top 5 = 38%.
- **Accessories:** 259 suppliers, top supplier only 6%. This is highly fragmented.

**Decision it supports.** Finished bikes, the high-performance core, depend on a single supplier, so Antonio should qualify a second source. Accessories are spread across many suppliers, so they are the natural place for a volume contract.

**Limitation / extra data.** The category mapping is our own, derived from item text and material prefix. Antonio would need a clean material hierarchy and quality/performance data per component before calling any supplier "best" for high-performance parts.